### **Entrenamiento y Ajuste de Hiperparámetros: Estrategia de Fusión Temprana (*Early Fusion*)**

Para las combinaciones de extractores seleccionadas que mayor rendimiento conjunto obtienen, se entrenan a continuación las arquitecturas multimodales resultantes de combinar dichos extractores mediante la estrategia de **fusión temprana**, llevando a cabo un ajuste de hiperparámetros mediante un **Grid Search**.

- Recordamos las combinaciones de extractores ganadoras seleccionadas:

    1. **`ResNet` + `Wav2Vec` + `BERT`**
    2. **`ResNet` + `MFCCs`(+`RMS`+`ZCR`) + `BERT`**
    3. **`ViT` + `Wav2Vec` + `BERT`** 


La arquitectura multimodal para esta estrategia de fusión temprana cuenta con los siguientes componentes:

1. **Extractores seleccionados por cada modalidad** (ya indicados).
2. **Adaptadores** independientes por cada modalidad que mapeen dichos vectores de características extraídos a un espacio latente común normalizado (constituidos por una LSTM + capa lineal, además de aplicar normalización, ReLU y dropout). 
3. **Fusión Temprana**: Concatenación directa de estos vectores obtenidos.
4. **Clasificador MLP**: Mismo clasificador explicado en el anterior *notebook*, constituido por las siguientes capas y en este mismo orden: `BatchNorm1d` (capa de normalización), `ReLU`, `Dropout`, capa lineal de compresión, `BatchNorm1d`, `ReLU`, `Dropout` y una última capa lineal para obtener el *logit* (resultado único en crudo de la red).



Los hiperparámetros a ajustar serán los siguientes:
- **Ventanas temporales**:
    - `audio_len`: 11 segundos o 7.
    - `video_frames`: 32 frames o 16.
    - **TEXTO**: 64 tokens o 32.

- **Configuración de la arquitectura**:
    - `proj_dim` (**default=512**): Dimensión de proyección de los adaptadores (espacio latente común).
    - `hidden_mlp` (**default=128**): Dimensión oculta del clasificador MLP final. No modificamos el número de capas (fijamos la profundidad de la red), sino que se ajusta el número de neuronas de la primera capa del clasificador (variamos la anchura o capacidad de la red).

    Como estos dos parámetros están directamente relacionados en la arquitectura multimodal, vamos a probar a ajustar las siguientes combinaciones:

    - `proj_dim` = 512 --> `hidden_mlp` = 256 y `hidden_mlp` = 128.
    - `proj_dim` = 256 --> `hidden_mlp` = 128 y `hidden_mlp` = 64.


- **Entrenamiento y regularización**: 
    - `lr` (**default=1e-4**): Learning Rate. Valores a ajustar: (`[1e-4, 5e-5, 1e-5]`).
    - `dropout` (**default=0.5**): Probabilidad de Dropout. Valores a ajustar: (`[0.3, 0.4, 0.5]`).

Los valores indicados para el ajuste por cada hiperparámetro se han extraído de la literatura (excepto las ventanas, extraídas del EDA). Se han limitado a una cantidad razonable para evitar la explosión combinatoria. 

Por otro lado, los hiperparámetros que hemos decidido **FIJAR** (NO los ajustamos) son:
- `epochs` (**default=20**): Número máximo de epochs.
- `patience` (**default=5**): Paciencia para Early Stopping.
- `pos_weight_mult`(**default=1.0**): Multiplicador para el peso de la clase positiva.
- `batch_size` (**default=32**): Tamaño del batch. Debido a su relación directa con el Learning Rate, se ha decidido fijarlo con 32 y ajustar Learning Rate. 
- `weight_decay` (**default=1e-2**): Weight decay (Regularización L2). El valor por defecto (1e-2) es el valor óptimo indicado en la literatura para el optimizador AdamW. 
- `hidden_lstm`: Se calcula y fija de forma dinámica en los adaptadores como paso intermedio (proporcional a la entrada del backbone y al `proj_dim`) para evitar cuellos de botella en la compresión temporal.

Para estos, se dejan los valores por defecto. 


Los modelos se entrenarán y ajustarán sobre:
- Particiones *train* y *dev* (para validación) del **Dataset Global Unificado (MELD + IEMOCAP)**.
- Particiones *train* y *dev* (para validación) del **Dataset Individual MELD (preprocesado)**.
- Particiones *train* y *dev* (para validación) del **Dataset Individual IEMOCAP (preprocesado)**.

Se obtendrá un modelo *ajustado* por cada uno de estos datasets, resultando en **9 arquitecturas multimodales** (con los extractores ya fijados).

In [ ]:
# Importamos las librerías necesarias:

import os
import json

-----

## **Dataset Global Unificado (MELD + IEMOCAP)**

In [ ]:
FUSIONES = ['early', 'late', 'attention']
VIDEO_BACKBONE = 'vit'
AUDIO_BACKBONE = 'wav2vec'
VENTANAS_VIDEO = [32, 16]
VENTANAS_AUDIO = [11, 7]
VENTANAS_TEXTO = ['roberta64', 'roberta32']
ARQ_COMB = [(512, 256), (512, 128), (256, 128), (256, 64)]
LRS = [1e-4, 5e-5, 1e-5]
DROPOUTS = [0.3, 0.4, 0.5]

RECALL_ESTRES_MINIMO = 0.45  # Filtro mínimo obligatorio para la clase estrés, si el modelo no alcanza este recall en estrés se descarta

mejor_f1_global = 0.0
mejor_config= None

for fusion in FUSIONES:
    for video_frames in VENTANAS_VIDEO:
        for audio_len in VENTANAS_AUDIO:
            for text_backbone in VENTANAS_TEXTO:
                for proj_dim, hidden_mlp in ARQ_COMB:
                    for lr in LRS:
                        for dropout in DROPOUTS:

                            nombre_base = (
                                f"global_{fusion}_{VIDEO_BACKBONE}{video_frames}_"
                                f"{AUDIO_BACKBONE}{audio_len}s_{text_backbone}_"
                                f"p{proj_dim}_h{hidden_mlp}_lr{lr}_do{dropout}"
                            )
                            pth_path  = f"pesos_modelo_estres_{nombre_base}.pth"
                            json_path = f"historial_estres_{nombre_base}.json"

                            !python train.py \
                                --train_dataset global \
                                --fusion        {fusion} \
                                --video         {VIDEO_BACKBONE} \
                                --video_frames  {video_frames} \
                                --audio         {AUDIO_BACKBONE} \
                                --audio_len     {audio_len} \
                                --text          {text_backbone} \
                                --proj_dim      {proj_dim} \
                                --hidden_mlp    {hidden_mlp} \
                                --lr            {lr} \
                                --dropout       {dropout}

                            # Extraemos ambas métricas del historial:
                            val_f1 = 0.0
                            val_recall = 0.0
                            if os.path.exists(json_path):
                                with open(json_path, 'r') as f:
                                    historial = json.load(f)
                                val_f1 = max(historial['val_f1'])
                                val_recall = max(historial['val_recall_estres'])

                            print(f"{nombre_base} --> F1: {val_f1:.4f} | Recall estrés: {val_recall:.4f}")

                            # Doble criterio: filtro de recall + selección por F1-Macro
                            supera_filtro = val_recall >= RECALL_ESTRES_MINIMO
                            mejora_f1     = val_f1 > mejor_f1_global

                            if supera_filtro and mejora_f1:
                                if mejor_config is not None:
                                    for archivo in [
                                        f"pesos_modelo_estres_{mejor_config}.pth",
                                        f"historial_estres_{mejor_config}.json"
                                    ]:
                                        if os.path.exists(archivo):
                                            os.remove(archivo)

                                mejor_f1_global = val_f1
                                mejor_config    = nombre_base
                                print(f"  --> NUEVO MEJOR MODELO (F1={mejor_f1_global:.4f} | Recall={val_recall:.4f}): {nombre_base}")
                            else:
                                if not supera_filtro:
                                    print(f"  --> DESCARTADO: Recall estrés ({val_recall:.4f}) < umbral ({RECALL_ESTRES_MINIMO})")
                                for archivo in [pth_path, json_path]:
                                    if os.path.exists(archivo):
                                        os.remove(archivo)

print(f"\nMEJOR CONFIGURACIÓN GLOBAL FINAL: {mejor_config}")
print(f"Mejor Val F1: {mejor_f1_global:.4f}")